In [ ]:
# @title 📦 Setup (run this cell first)
%matplotlib inline
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from IPython.display import display, HTML, clear_output

# Matplotlib style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.facecolor'] = 'white'

# Okabe-Ito palette
COLORS = {
    'text': '#E69F00', 'image': '#0072B2', 'connection': '#56B4E9',
    'failure': '#D55E00', 'crossmodal': '#CC79A7', 'neutral': '#999999'
}

print('✅ Setup complete.')

In [ ]:
# @title 🔧 Helper Functions (collapsed)

SHAPES = ['circle', 'square', 'triangle']
COLORS_MAP = {
    'red': (230, 25, 25),
    'blue': (25, 25, 230),
    'green': (25, 204, 25),
}
SIZES = {'small': 5, 'large': 10}
POSITIONS = {
    'center': (16, 16),
    'offset': [(8, 8), (8, 24), (24, 8), (24, 24)],
}

def generate_shape(shape, color, size='large', position='center', img_size=32):
    img = Image.new('RGB', (img_size, img_size), (0, 0, 0))
    draw = ImageDraw.Draw(img)
    rgb = COLORS_MAP[color]
    radius = SIZES[size]
    if position == 'center':
        cx, cy = 16, 16
    else:
        rng = np.random.RandomState()
        cx, cy = POSITIONS['offset'][rng.randint(len(POSITIONS['offset']))]
    if shape == 'circle':
        draw.ellipse([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'square':
        draw.rectangle([cx-radius, cy-radius, cx+radius, cy+radius], fill=rgb)
    elif shape == 'triangle':
        pts = [(cx, cy-radius), (cx-radius, cy+radius), (cx+radius, cy+radius)]
        draw.polygon(pts, fill=rgb)
    return np.array(img, dtype=np.float32) / 255.0

def generate_dataset(include_size=False, include_position=False):
    sizes = list(SIZES.keys()) if include_size else ['large']
    positions = ['center', 'offset'] if include_position else ['center']
    dataset = []
    for color in COLORS_MAP:
        for shape in SHAPES:
            for sz in sizes:
                for pos in positions:
                    img = generate_shape(shape, color, size=sz, position=pos)
                    label = f"{sz} {color} {shape}" if include_size else f"{color} {shape}"
                    dataset.append((img, label))
    return dataset

class ShapeDataset(torch.utils.data.Dataset):
    def __init__(self, include_size=False, include_position=False):
        raw = generate_dataset(include_size=include_size, include_position=include_position)
        self.images, self.labels, self.label_texts = [], [], []
        unique_labels = []
        for _, label in raw:
            if label not in unique_labels:
                unique_labels.append(label)
        self.label_to_idx = {l: i for i, l in enumerate(unique_labels)}
        for img, label in raw:
            tensor = torch.from_numpy(img).permute(2, 0, 1)
            self.images.append(tensor)
            self.labels.append(self.label_to_idx[label])
            self.label_texts.append(label)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx], self.label_texts[idx]

# --- Visualization helpers ---
def plot_image_grid(images, titles=None, ncols=3, figsize=None, suptitle=None):
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    if figsize is None:
        figsize = (ncols * 2.5, nrows * 2.5)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, torch.Tensor):
                if img.dim() == 3 and img.shape[0] in (1, 3):
                    img = img.permute(1, 2, 0)
                img = img.detach().cpu().numpy()
            img = np.clip(img, 0, 1)
            ax.imshow(img)
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=12)
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_denoising_filmstrip(images, steps=None, title='Denoising Process'):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 2.5))
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        if isinstance(img, torch.Tensor):
            if img.dim() == 3 and img.shape[0] in (1, 3):
                img = img.permute(1, 2, 0)
            img = img.detach().cpu().numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        label = f't={steps[i]}' if steps else f'Step {i}'
        ax.set_title(label, fontsize=10)
        ax.axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

def plot_training_loss(losses, title='Training Loss', color=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    c = color or COLORS['image']
    ax.plot(losses, color=c, linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig

def plot_report_card(images, prompts, color_scores, shape_scores):
    n = len(images)
    ncols = min(n, 3)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3.5))
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            img = images[i]
            if isinstance(img, torch.Tensor):
                if img.dim() == 3 and img.shape[0] in (1, 3):
                    img = img.permute(1, 2, 0)
                img = img.detach().cpu().numpy()
            img = np.clip(img, 0, 1)
            ax.imshow(img)
            c_sym = '✅' if color_scores[i] else '❌'
            s_sym = '✅' if shape_scores[i] else '❌'
            ax.set_title(f'{prompts[i]}\nColor {c_sym}  Shape {s_sym}', fontsize=11)
        ax.axis('off')
    fig.suptitle('Report Card', fontsize=14, fontweight='bold')
    fig.tight_layout()
    return fig

print('✅ Helpers loaded.')

In [ ]:
# @title 🏗️ Models (collapsed)

VOCAB = [
    'red', 'blue', 'green', 'circle', 'square', 'triangle',
    'small', 'large', 'center', 'offset', '<pad>', '<unk>'
]
WORD_TO_IDX = {w: i for i, w in enumerate(VOCAB)}

def tokenize(text, max_len=4):
    tokens = []
    for word in text.lower().split():
        tokens.append(WORD_TO_IDX.get(word, WORD_TO_IDX['<unk>']))
    if len(tokens) < max_len:
        tokens += [WORD_TO_IDX['<pad>']] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    return torch.tensor(tokens, dtype=torch.long)

class TextEncoder(nn.Module):
    def __init__(self, vocab_size=12, embed_dim=32, hidden_dim=64, out_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=WORD_TO_IDX['<pad>'])
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
    def forward(self, token_ids):
        x = self.embedding(token_ids)
        mask = (token_ids != WORD_TO_IDX['<pad>']).unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

class ImageEncoder(nn.Module):
    def __init__(self, input_dim=3072, out_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, out_dim)
    def forward(self, images):
        x = images.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.normalize(x, dim=-1)

class MiniCLIP(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder = TextEncoder()
        self.image_encoder = ImageEncoder()
    def forward(self, images, token_ids):
        return self.text_encoder(token_ids), self.image_encoder(images)
    def compute_loss(self, text_emb, image_emb, tau=0.07):
        logits = torch.matmul(text_emb, image_emb.t()) / tau
        labels = torch.arange(len(logits), device=logits.device)
        return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2

class NaiveMLP(nn.Module):
    """Intentional failure model: no spatial inductive bias. ~3.2M params."""
    def __init__(self, text_emb_dim=32, time_emb_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(3072 + time_emb_dim + text_emb_dim, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 3072)
    def forward(self, x, t_emb, text_emb):
        B = x.shape[0]
        flat = x.flatten(1)
        inp = torch.cat([flat, t_emb, text_emb], dim=1)
        h = F.relu(self.fc1(inp))
        h = F.relu(self.fc2(h))
        return self.fc3(h).view(B, 3, 32, 32)

class CrossAttention(nn.Module):
    def __init__(self, img_dim=128, text_dim=32, head_dim=32):
        super().__init__()
        self.W_q = nn.Linear(img_dim, head_dim)
        self.W_k = nn.Linear(text_dim, head_dim)
        self.W_v = nn.Linear(text_dim, head_dim)
        self.W_out = nn.Linear(head_dim, img_dim)
        self.scale = head_dim ** -0.5
    def forward(self, image_features, text_features):
        Q = self.W_q(image_features)
        K = self.W_k(text_features)
        V = self.W_v(text_features)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn, dim=-1)
        out = torch.matmul(attn_weights, V)
        out = self.W_out(out)
        return out, attn_weights

class MicroUNet(nn.Module):
    """Minimal UNet with cross-attention at bottleneck. ~150K params."""
    def __init__(self, time_emb_dim=32, text_emb_dim=32):
        super().__init__()
        self.enc1 = nn.Conv2d(3, 32, 3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.time_mlp = nn.Sequential(nn.Linear(time_emb_dim, 128), nn.ReLU())
        self.cross_attn = CrossAttention(img_dim=128, text_dim=text_emb_dim, head_dim=32)
        self.dec1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(128, 32, 4, stride=2, padding=1)
        self.final = nn.Conv2d(64, 3, 3, padding=1)
    def forward(self, x, t_emb, text_emb):
        out, _ = self.forward_with_attention(x, t_emb, text_emb)
        return out
    def forward_with_attention(self, x, t_emb, text_emb):
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2))
        t = self.time_mlp(t_emb)
        e3 = e3 + t[:, :, None, None]
        B, C, H, W = e3.shape
        patches = e3.permute(0, 2, 3, 1).reshape(B, H * W, C)
        text_feat = text_emb.unsqueeze(1)
        attn_out, attn_weights = self.cross_attn(patches, text_feat)
        e3 = attn_out.reshape(B, H, W, C).permute(0, 3, 1, 2)
        d1 = F.relu(self.dec1(e3))
        d1 = torch.cat([d1, e2], dim=1)
        d2 = F.relu(self.dec2(d1))
        d2 = torch.cat([d2, e1], dim=1)
        out = self.final(d2)
        return out, attn_weights

class NoiseScheduler:
    """Cosine noise schedule with T=50 steps."""
    def __init__(self, T=50, s=0.008):
        self.T = T
        self.s = s
        steps = torch.arange(T + 1, dtype=torch.float64)
        f = torch.cos((steps / T + s) / (1 + s) * math.pi / 2) ** 2
        alpha_bars = f / f[0]
        alpha_bars = alpha_bars.clamp(min=1e-5, max=1.0)
        betas = 1 - alpha_bars[1:] / alpha_bars[:-1]
        betas = betas.clamp(min=0.0001, max=0.999)
        self.betas = betas.float()
        self.alphas = (1 - self.betas).float()
        self.alpha_bars = torch.cumprod(self.alphas, dim=0).float()

    def add_noise(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)
        alpha_bar_t = self.alpha_bars[t]
        while alpha_bar_t.dim() < x_0.dim():
            alpha_bar_t = alpha_bar_t.unsqueeze(-1)
        return torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise

    def sample_step(self, x_t, t, predicted_noise):
        beta_t = self.betas[t]
        alpha_t = self.alphas[t]
        alpha_bar_t = self.alpha_bars[t]
        while beta_t.dim() < x_t.dim():
            beta_t = beta_t.unsqueeze(-1)
            alpha_t = alpha_t.unsqueeze(-1)
            alpha_bar_t = alpha_bar_t.unsqueeze(-1)
        mean = (1 / torch.sqrt(alpha_t)) * (x_t - (beta_t / torch.sqrt(1 - alpha_bar_t)) * predicted_noise)
        if isinstance(t, int) and t == 0:
            return mean
        if isinstance(t, torch.Tensor) and (t == 0).all():
            return mean
        noise = torch.randn_like(x_t)
        sigma = torch.sqrt(beta_t)
        return mean + sigma * noise

    @staticmethod
    def get_time_embedding(t, dim=32):
        if isinstance(t, int):
            t = torch.tensor([t], dtype=torch.float32)
        elif isinstance(t, torch.Tensor) and t.dim() == 0:
            t = t.unsqueeze(0).float()
        else:
            t = t.float()
        half_dim = dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, dtype=torch.float32) * -emb)
        emb = t.unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)

print('✅ Models loaded.')

In [ ]:
# @title 🔄 Training Functions (collapsed)

def train_clip(model, dataset, epochs=100, lr=1e-3, tau=0.07):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    texts = [dataset[i][2] for i in range(len(dataset))]
    token_ids = torch.stack([tokenize(t) for t in texts])
    for epoch in range(epochs):
        optimizer.zero_grad()
        text_emb, image_emb = model(images, token_ids)
        loss = model.compute_loss(text_emb, image_emb, tau=tau)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 20 == 0:
            print(f'CLIP Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
    return losses

def train_denoiser(model, dataset, scheduler, text_encoder, epochs=300, lr=1e-4, is_mlp=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    texts = [dataset[i][2] for i in range(len(dataset))]
    token_ids = torch.stack([tokenize(t) for t in texts])
    with torch.no_grad():
        text_embs = text_encoder(token_ids)
    for epoch in range(epochs):
        optimizer.zero_grad()
        B = len(images)
        t = torch.randint(0, scheduler.T, (B,))
        noise = torch.randn_like(images)
        x_t = scheduler.add_noise(images, t, noise)
        t_emb = scheduler.get_time_embedding(t)
        predicted_noise = model(x_t, t_emb, text_embs)
        loss = F.mse_loss(predicted_noise, noise)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 50 == 0:
            print(f'Denoiser Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
    return losses

@torch.no_grad()
def generate_image(model, scheduler, text_emb, null_text_emb=None,
                   guidance_scale=7.5, steps=50, seed=42):
    torch.manual_seed(seed)
    x = torch.randn(1, 3, 32, 32)
    intermediates = []
    save_steps = set([steps - 1] + [int(i * (steps - 1) / 7) for i in range(8)])
    for t_idx in reversed(range(steps)):
        t = torch.tensor([t_idx])
        t_emb = scheduler.get_time_embedding(t)
        if null_text_emb is not None and guidance_scale != 1.0:
            noise_cond = model(x, t_emb, text_emb)
            noise_uncond = model(x, t_emb, null_text_emb)
            predicted_noise = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        else:
            predicted_noise = model(x, t_emb, text_emb)
        x = scheduler.sample_step(x, t_idx, predicted_noise)
        if t_idx in save_steps:
            intermediates.append(x[0].clone().clamp(0, 1))
    intermediates.reverse()
    final = x[0].clamp(0, 1)
    return final, intermediates

def color_accuracy(image, expected_color):
    if isinstance(image, torch.Tensor):
        if image.dim() == 3 and image.shape[0] == 3:
            image = image.permute(1, 2, 0)
        image = image.detach().cpu().numpy()
    brightness = image.mean(axis=2)
    mask = brightness > 0.1
    if mask.sum() == 0:
        return False
    avg_color = image[mask].mean(axis=0)
    channel_map = {'red': 0, 'blue': 2, 'green': 1}
    return np.argmax(avg_color) == channel_map[expected_color]

def shape_accuracy(image, expected_shape):
    if isinstance(image, torch.Tensor):
        if image.dim() == 3 and image.shape[0] == 3:
            image = image.permute(1, 2, 0)
        image = image.detach().cpu().numpy()
    brightness = image.mean(axis=2)
    gen_mask = (brightness > 0.15).astype(np.float32)
    if gen_mask.sum() < 5:
        return False
    templates = {}
    for shape in ['circle', 'square', 'triangle']:
        tpl = Image.new('L', (32, 32), 0)
        draw = ImageDraw.Draw(tpl)
        r, cx, cy = 10, 16, 16
        if shape == 'circle':
            draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=255)
        elif shape == 'square':
            draw.rectangle([cx-r, cy-r, cx+r, cy+r], fill=255)
        elif shape == 'triangle':
            draw.polygon([(cx, cy-r), (cx-r, cy+r), (cx+r, cy+r)], fill=255)
        templates[shape] = np.array(tpl, dtype=np.float32) / 255.0
    gen_norm = gen_mask - gen_mask.mean()
    gen_std = gen_norm.std()
    if gen_std < 1e-6:
        return False
    best_shape, best_corr = None, -2.0
    for shape, tpl_mask in templates.items():
        tpl_norm = tpl_mask - tpl_mask.mean()
        tpl_std = tpl_norm.std()
        if tpl_std < 1e-6:
            continue
        corr = (gen_norm * tpl_norm).sum() / (gen_std * tpl_std * gen_mask.size)
        if corr > best_corr:
            best_corr = corr
            best_shape = shape
    return best_shape == expected_shape

# --- Load pre-trained CLIP (needed for text embeddings) ---
dataset = ShapeDataset()
scheduler = NoiseScheduler(T=50)
clip_model = MiniCLIP()

import os
weights_loaded = False
# Try to load pre-trained weights
for path in ['weights/full_pipeline.pt', '../weights/full_pipeline.pt',
             '/content/weights/full_pipeline.pt']:
    if os.path.exists(path):
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
        clip_model.text_encoder.load_state_dict(
            {k: v.float() for k, v in checkpoint['clip_text'].items()})
        clip_model.image_encoder.load_state_dict(
            {k: v.float() for k, v in checkpoint['clip_image'].items()})
        weights_loaded = True
        print(f'✅ Loaded pre-trained CLIP from {path}')
        break

if not weights_loaded:
    print('⚠️ Pre-trained weights not found. Training CLIP from scratch (~30 sec)...')
    clip_model = MiniCLIP()
    train_clip(clip_model, dataset, epochs=100)
    print('✅ CLIP trained.')

clip_model.eval()
print('✅ Training functions loaded.')

In [ ]:
# Runtime check
try:
    _ = len(COLORS)
    print('✅ Ready to go!')
except NameError:
    print('⚠️ Please run the setup cells above (grey cells 1-4).')
    print('   Click the ▶ button on each grey cell at the top.')

# 🚗 Notebook 3: The Engine — Diffusion

**How does a model turn random noise into a picture?**

In the last notebook, CLIP learned to connect text and images. But CLIP only *understands* — it doesn’t *create*. Now we build the **engine** that actually generates images.

The secret? **Learn to remove noise.** If you can remove a little noise, you can do it 50 times in a row and turn pure static into a picture.

In [ ]:
# DEMO: Forward process (destruction) vs Reverse process (reconstruction)
# Watch how noise destroys an image, then how the model rebuilds it.

demo_img = dataset[0][0]  # red circle
demo_prompt = dataset[0][2]

# Forward: clean -> noise (8 frames)
forward_steps = [0, 7, 14, 21, 28, 35, 42, 49]
torch.manual_seed(0)
fixed_noise = torch.randn_like(demo_img.unsqueeze(0))
forward_frames = []
for t in forward_steps:
    noisy = scheduler.add_noise(demo_img.unsqueeze(0), torch.tensor([t]), fixed_noise)
    forward_frames.append(noisy[0].clamp(0, 1))

# Reverse: use pre-trained model if available
unet_demo = MicroUNet()
unet_loaded = False
for path in ['weights/full_pipeline.pt', '../weights/full_pipeline.pt',
             '/content/weights/full_pipeline.pt']:
    if os.path.exists(path):
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        unet_demo.load_state_dict({k: v.float() for k, v in ckpt['unet'].items()})
        unet_loaded = True
        break

if unet_loaded:
    unet_demo.eval()
    tokens = tokenize(demo_prompt)
    text_emb = clip_model.text_encoder(tokens.unsqueeze(0))
    null_tokens = tokenize('<pad> <pad>')
    null_emb = clip_model.text_encoder(null_tokens.unsqueeze(0))
    _, reverse_frames = generate_image(unet_demo, scheduler, text_emb, null_emb, seed=42)
else:
    reverse_frames = list(reversed(forward_frames))  # placeholder

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(16, 6))
ax_top.axis('off'); ax_bot.axis('off')

fig_fwd = plot_denoising_filmstrip(forward_frames, steps=forward_steps,
                                    title='⬇️ Forward Process: Destruction (clean → noise)')
plt.show()
fig_rev = plot_denoising_filmstrip(reverse_frames,
                                    steps=list(range(0, 50, 50//len(reverse_frames)))[:len(reverse_frames)],
                                    title='⬆️ Reverse Process: Reconstruction (noise → image)')
plt.show()
print('Left: destruction. Right: reconstruction.')
print('The model only learns the RIGHT side — how to remove noise, one step at a time.')

---

> **🧭 Orient:** *"The driver learns by being dropped at random locations and finding the destination, thousands of times."*

The diffusion model is like a driver who learns to navigate by practicing over and over. Each time, we drop the driver at a random point between the destination and complete chaos. After enough practice, the driver can find the way home from *anywhere* — even from pure noise.

---

In [ ]:
# INTERACT #1: Forward Process Explorer
# Drag the slider to see how noise gradually destroys an image.

# Pre-compute all 51 noisy versions for instant response
base_image = dataset[0][0]  # red circle
torch.manual_seed(0)
fixed_noise_fwd = torch.randn_like(base_image.unsqueeze(0))
noisy_cache = {}
for t in range(scheduler.T + 1):
    t_val = min(t, scheduler.T - 1)
    noisy = scheduler.add_noise(base_image.unsqueeze(0), torch.tensor([t_val]), fixed_noise_fwd)
    noisy_cache[t] = noisy[0].clamp(0, 1)

@interact(timestep=IntSlider(min=0, max=50, step=1, value=0, description='Timestep t:'))
def explore_forward(timestep):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Left: noisy image
    img = noisy_cache[timestep].permute(1, 2, 0).numpy()
    axes[0].imshow(np.clip(img, 0, 1))
    axes[0].set_title(f'Image at t={timestep}', fontsize=13)
    axes[0].axis('off')

    # Middle: noise schedule (beta_t)
    ts = np.arange(scheduler.T)
    axes[1].plot(ts, scheduler.betas.numpy(), color=COLORS['failure'], linewidth=2)
    if timestep > 0 and timestep <= scheduler.T:
        axes[1].axvline(x=timestep-1, color=COLORS['neutral'], linestyle='--', linewidth=1.5)
        axes[1].scatter([timestep-1], [scheduler.betas[timestep-1].item()],
                       color=COLORS['failure'], s=80, zorder=5)
    axes[1].set_xlabel('Timestep', fontsize=12)
    axes[1].set_ylabel('\u03b2\u209c (noise added)', fontsize=12)
    axes[1].set_title('Noise Schedule (\u03b2\u209c)', fontsize=13)
    axes[1].grid(True, alpha=0.3)

    # Right: cumulative signal (alpha_bar_t)
    axes[2].plot(ts, scheduler.alpha_bars.numpy(), color=COLORS['connection'], linewidth=2)
    if timestep > 0 and timestep <= scheduler.T:
        axes[2].axvline(x=timestep-1, color=COLORS['neutral'], linestyle='--', linewidth=1.5)
        axes[2].scatter([timestep-1], [scheduler.alpha_bars[timestep-1].item()],
                       color=COLORS['connection'], s=80, zorder=5)
    axes[2].set_xlabel('Timestep', fontsize=12)
    axes[2].set_ylabel('\u0101\u0305\u209c (signal remaining)', fontsize=12)
    axes[2].set_title('Signal Remaining (\u0101\u0305\u209c)', fontsize=13)
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

    if timestep == 0:
        print('\u2190 At t=0, 100% signal. The image is perfectly clean.')
    elif timestep < 25:
        pct = scheduler.alpha_bars[timestep-1].item() * 100
        print(f'\u2190 At t={timestep}, {pct:.0f}% signal remains. You can still see the shape.')
    elif timestep == 50:
        print('\u2190 At t=50, ~0% signal. Pure noise. The image is completely gone.')
    else:
        pct = scheduler.alpha_bars[timestep-1].item() * 100
        print(f'\u2190 At t={timestep}, only {pct:.0f}% signal remains.')

In [ ]:
# INTERACT #2: Step-by-Step Denoising
# Walk through the reverse process manually.

if unet_loaded:
    # Pre-compute denoising at 10 evenly spaced steps
    tokens = tokenize(demo_prompt)
    text_emb_demo = clip_model.text_encoder(tokens.unsqueeze(0))
    null_tokens = tokenize('<pad> <pad>')
    null_emb_demo = clip_model.text_encoder(null_tokens.unsqueeze(0))

    denoise_cache = {}
    torch.manual_seed(42)
    x = torch.randn(1, 3, 32, 32)
    denoise_cache[50] = x[0].clone().clamp(0, 1)
    for t_idx in reversed(range(50)):
        t = torch.tensor([t_idx])
        t_emb = scheduler.get_time_embedding(t)
        noise_cond = unet_demo(x, t_emb, text_emb_demo)
        noise_uncond = unet_demo(x, t_emb, null_emb_demo)
        pred = noise_uncond + 7.5 * (noise_cond - noise_uncond)
        x = scheduler.sample_step(x, t_idx, pred)
        denoise_cache[t_idx] = x[0].clone().clamp(0, 1)

    @interact(step=IntSlider(min=0, max=50, step=5, value=50, description='Denoise from:'))
    def explore_denoise(step):
        fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
        # Current noisy image
        img = denoise_cache[step].permute(1, 2, 0).numpy()
        axes[0].imshow(np.clip(img, 0, 1))
        axes[0].set_title(f'Current (t={step})', fontsize=12)
        axes[0].axis('off')

        # 5 steps later
        next_step = max(0, step - 5)
        img2 = denoise_cache[next_step].permute(1, 2, 0).numpy()
        axes[1].imshow(np.clip(img2, 0, 1))
        axes[1].set_title(f'After 5 steps (t={next_step})', fontsize=12)
        axes[1].axis('off')

        # Final result
        img3 = denoise_cache[0].permute(1, 2, 0).numpy()
        axes[2].imshow(np.clip(img3, 0, 1))
        axes[2].set_title('Final (t=0)', fontsize=12)
        axes[2].axis('off')

        fig.tight_layout()
        plt.show()
        if step == 50:
            print('Starting from pure noise. Drag the slider left to denoise!')
        elif step == 0:
            print('\u2705 Fully denoised! The red circle has emerged from noise.')
        else:
            print(f'Denoising in progress... {50-step}/50 steps done.')
else:
    print('\u26a0\ufe0f Pre-trained UNet not available. This interactive demo requires weights.')
    print('   The model will be trained later in this notebook.')

In [ ]:
# BUILD: The cosine noise schedule
# This defines HOW MUCH noise to add at each timestep.

def cosine_schedule(T=50, s=0.008):
    """Cosine noise schedule: smooth transition from clean to noisy."""
    steps = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos((steps / T + s) / (1 + s) * math.pi / 2) ** 2
    alpha_bars = (f / f[0]).clamp(min=1e-5, max=1.0)
    betas = (1 - alpha_bars[1:] / alpha_bars[:-1]).clamp(0.0001, 0.999)
    return betas.float(), torch.cumprod(1 - betas, dim=0).float()

betas, alpha_bars = cosine_schedule(T=50)
print(f'Steps: {len(betas)}')
print(f'Signal at t=0:  {alpha_bars[0]:.4f} (almost all signal)')
print(f'Signal at t=25: {alpha_bars[24]:.4f} (half-way)')
print(f'Signal at t=49: {alpha_bars[48]:.4f} (almost pure noise)')

In [ ]:
# BUILD: The forward noise function
# This is how we CREATE training data: take a clean image, add noise.

def add_noise(x_0, alpha_bar_t, noise=None):
    """Add noise to a clean image: x_t = sqrt(alpha_bar) * x_0 + sqrt(1-alpha_bar) * noise"""
    if noise is None:
        noise = torch.randn_like(x_0)
    return torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise

# Quick demo: add noise at different levels
test_image = dataset[0][0].unsqueeze(0)  # red circle
print('Clean image -> noisy image at different timesteps:')
for t in [0, 10, 25, 40, 49]:
    ab = alpha_bars[t]
    print(f'  t={t:2d}: signal={ab:.3f}, noise={1-ab:.3f}')

In [ ]:
# BUILD: The noise prediction loss
# The model's job: given a noisy image and timestep, predict the noise.

def noise_prediction_loss(model, x_0, t, text_emb, scheduler):
    """MSE between actual noise and model's prediction."""
    noise = torch.randn_like(x_0)                   # Sample random noise
    x_t = scheduler.add_noise(x_0, t, noise)        # Create noisy image
    t_emb = scheduler.get_time_embedding(t)          # Encode timestep
    predicted = model(x_t, t_emb, text_emb)          # Model predicts noise
    return F.mse_loss(predicted, noise)               # Compare prediction to truth

print('The training loop repeats this for thousands of random (image, timestep) pairs.')
print('Over time, the model learns to recognize and remove noise at any level.')

In [ ]:
# BUILD: Train the NaiveMLP denoiser (150 epochs)
# This is our FIRST attempt. It will fail in an interesting way.

print('Training NaiveMLP (a simple fully-connected network)...')
print(f'Parameters: {sum(p.numel() for p in NaiveMLP().parameters()):,} (~3.2 million)')
print()

mlp_model = NaiveMLP()
mlp_losses = train_denoiser(mlp_model, dataset, scheduler,
                             clip_model.text_encoder, epochs=150, lr=1e-4)

fig = plot_training_loss(mlp_losses, title='NaiveMLP Training Loss', color=COLORS['failure'])
plt.show()
print(f'Final loss: {mlp_losses[-1]:.4f}')

In [ ]:
# STUMBLE: Generate all 9 classes with the MLP
# Something is wrong... can you see it?

mlp_model.eval()
prompts = [dataset[i][2] for i in range(len(dataset))]
null_tokens = tokenize('<pad> <pad>')
null_emb = clip_model.text_encoder(null_tokens.unsqueeze(0))

mlp_images, mlp_color_ok, mlp_shape_ok = [], [], []
for prompt in prompts:
    tokens = tokenize(prompt)
    t_emb = clip_model.text_encoder(tokens.unsqueeze(0))
    img, _ = generate_image(mlp_model, scheduler, t_emb, null_emb, guidance_scale=7.5, seed=42)
    mlp_images.append(img)
    parts = prompt.split()
    mlp_color_ok.append(color_accuracy(img, parts[0]))
    mlp_shape_ok.append(shape_accuracy(img, parts[1]))

fig = plot_report_card(mlp_images, prompts, mlp_color_ok, mlp_shape_ok)
plt.show()

print(f'\nColor accuracy: {sum(mlp_color_ok)}/{len(mlp_color_ok)}')
print(f'Shape accuracy: {sum(mlp_shape_ok)}/{len(mlp_shape_ok)}')
print()
print('\u26a0\ufe0f Colors are roughly right, but shapes are blobs!')
print('   The MLP can learn "redness" but not "circle-ness." Why?')

In [ ]:
# Shift Experiment: Why does the MLP fail at shapes?
# Let's shift an image 3 pixels right and see what happens.

test_img = dataset[0][0].unsqueeze(0)  # red circle
shifted_img = torch.roll(test_img, 3, dims=-1)  # shift 3px right

# Add the same noise to both
t = torch.tensor([25])
torch.manual_seed(0)
noise = torch.randn_like(test_img)
noisy_orig = scheduler.add_noise(test_img, t, noise)
noisy_shifted = scheduler.add_noise(shifted_img, t, noise)

# MLP predictions
t_emb = scheduler.get_time_embedding(t)
text_emb_test = clip_model.text_encoder(tokenize('red circle').unsqueeze(0))
with torch.no_grad():
    pred_orig = mlp_model(noisy_orig, t_emb, text_emb_test)
    pred_shifted = mlp_model(noisy_shifted, t_emb, text_emb_test)

# How different are the predictions?
diff = (pred_orig - pred_shifted).abs().mean().item()

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for ax, img, title in [
    (axes[0, 0], test_img[0], 'Original'),
    (axes[0, 1], shifted_img[0], 'Shifted 3px right'),
    (axes[1, 0], pred_orig[0], 'MLP prediction (original)'),
    (axes[1, 1], pred_shifted[0], 'MLP prediction (shifted)'),
]:
    im = img.permute(1, 2, 0).detach().numpy()
    ax.imshow(np.clip(im, 0, 1))
    ax.set_title(title, fontsize=12)
    ax.axis('off')

fig.suptitle(f'MLP Shift Test (prediction difference: {diff:.4f})', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print(f'The MLP\'s prediction changed drastically when we shifted by just 3 pixels!')
print(f'Average prediction difference: {diff:.4f}')
print(f'The MLP treats each pixel independently. It has NO concept of "nearby."')

---

> **💡 Aha!** The MLP treats each pixel independently. It has no concept of "nearby." A convolutional network has spatial structure built in — it knows that pixel (5,5) is next to pixel (5,6).
>
> **Co-pilot metaphor:** *Imagine a driver who can’t see through the windshield — only reads instrument dials. They can tell you the temperature (color) but not where the road goes (shape). A convolutional network gives the driver a windshield.*

---

In [ ]:
# FIX: Meet the MicroUNet — a convolutional denoiser with spatial awareness

print('📊 MicroUNet Architecture')
print('=' * 50)
print()
print('🔽 Encoder (sees bigger picture at each step):')
print('   Conv2d(3\u219232)     32\u00d732  \u2190 Full resolution')
print('   Conv2d(32\u219264)    16\u00d716  \u2190 Half resolution')
print('   Conv2d(64\u2192128)    8\u00d78   \u2190 Quarter resolution')
print()
print('🔄 Bottleneck:')
print('   + Time embedding (when in the process are we?)')
print('   + Cross-attention (what does the text say?)')
print()
print('🔼 Decoder (rebuilds with skip connections):')
print('   ConvT2d(128\u219264)   16\u00d716  + skip from encoder')
print('   ConvT2d(128\u219232)   32\u00d732  + skip from encoder')
print('   Conv2d(64\u21923)      32\u00d732  \u2190 Final prediction')
print()

unet = MicroUNet()
param_count = sum(p.numel() for p in unet.parameters())
print(f'Total parameters: {param_count:,}')
print(f'\nCompare: MLP had {sum(p.numel() for p in NaiveMLP().parameters()):,} parameters')
print(f'The UNet is SMALLER but SMARTER \u2014 convolutions share knowledge across positions.')

In [ ]:
# FIX: Train MicroUNet (300 epochs)
# This takes ~2-3 minutes on CPU. Watch the loss drop!

print('Training MicroUNet (convolutional denoiser)...')
print(f'Parameters: {sum(p.numel() for p in unet.parameters()):,}')
print()

unet_losses = train_denoiser(unet, dataset, scheduler,
                              clip_model.text_encoder, epochs=300, lr=1e-4)

# Compare loss curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mlp_losses, color=COLORS['failure'], linewidth=2, label=f'NaiveMLP (150 epochs)', alpha=0.7)
ax.plot(unet_losses, color=COLORS['connection'], linewidth=2, label=f'MicroUNet (300 epochs)')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss: MLP vs UNet', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f'MLP final loss:  {mlp_losses[-1]:.4f}')
print(f'UNet final loss: {unet_losses[-1]:.4f}')
print(f'The UNet learns much more effectively despite having fewer parameters!')

In [ ]:
# Generate all 9 classes with MicroUNet + report card
# Compare side-by-side with the MLP results!

unet.eval()
unet_images, unet_color_ok, unet_shape_ok = [], [], []
for prompt in prompts:
    tokens = tokenize(prompt)
    t_emb = clip_model.text_encoder(tokens.unsqueeze(0))
    img, _ = generate_image(unet, scheduler, t_emb, null_emb, guidance_scale=7.5, seed=42)
    unet_images.append(img)
    parts = prompt.split()
    unet_color_ok.append(color_accuracy(img, parts[0]))
    unet_shape_ok.append(shape_accuracy(img, parts[1]))

# MLP results
fig_mlp = plot_report_card(mlp_images, prompts, mlp_color_ok, mlp_shape_ok)
fig_mlp.suptitle('❌ NaiveMLP Results (no spatial awareness)', fontsize=14, fontweight='bold',
                  color=COLORS['failure'])
plt.show()

# UNet results
fig_unet = plot_report_card(unet_images, prompts, unet_color_ok, unet_shape_ok)
fig_unet.suptitle('✅ MicroUNet Results (convolutional)', fontsize=14, fontweight='bold',
                   color=COLORS['connection'])
plt.show()

print(f'\nMLP  - Color: {sum(mlp_color_ok)}/9, Shape: {sum(mlp_shape_ok)}/9')
print(f'UNet - Color: {sum(unet_color_ok)}/9, Shape: {sum(unet_shape_ok)}/9')
print(f'\nThe convolutional architecture makes all the difference for shapes!')

In [ ]:
# Shift experiment on MicroUNet: spatial equivariance!

with torch.no_grad():
    pred_orig_unet = unet(noisy_orig, t_emb, text_emb_test)
    pred_shifted_unet = unet(noisy_shifted, t_emb, text_emb_test)

diff_unet = (pred_orig_unet - pred_shifted_unet).abs().mean().item()
# Also compute shifted prediction vs shifted original prediction
pred_orig_shifted = torch.roll(pred_orig_unet, 3, dims=-1)
diff_equivariant = (pred_orig_shifted - pred_shifted_unet).abs().mean().item()

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for ax, img, title in [
    (axes[0, 0], test_img[0], 'Original'),
    (axes[0, 1], shifted_img[0], 'Shifted 3px right'),
    (axes[1, 0], pred_orig_unet[0], 'UNet prediction (original)'),
    (axes[1, 1], pred_shifted_unet[0], 'UNet prediction (shifted)'),
]:
    im = img.permute(1, 2, 0).detach().numpy()
    ax.imshow(np.clip(im, 0, 1))
    ax.set_title(title, fontsize=12)
    ax.axis('off')

fig.suptitle(f'UNet Shift Test', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print(f'MLP  prediction change after shift: {diff:.4f} (large = position-dependent)')
print(f'UNet prediction change after shift: {diff_unet:.4f}')
print(f'UNet equivariance error: {diff_equivariant:.4f} (small = shift-equivariant)')
print()
print('The UNet\'s prediction shifts WITH the image.')
print('Convolutions give the model spatial understanding for free.')

---

<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0;">

### 💼 Real-World Story: The Deepfake Dilemma

In 2023, AI-generated images were submitted as evidence in legal proceedings. The images looked photorealistic because **that is exactly what the model was trained to do** — minimize the difference between generated and real images.

**Formal diagnosis:** The training objective is `p(generated) ≈ p(real)`. Photorealism isn’t a bug — it’s the entire point of the loss function. Detection requires looking at *spectral artifacts* (patterns in the frequency domain) that humans can’t see but specialized classifiers can.

**As a decision-maker:** Ask your vendor: *"What detection mechanisms do you provide? Do you embed invisible watermarks?"*

</div>

---

---

<div style="background-color: #FFF3CD; border-left: 4px solid #E69F00; padding: 15px; margin: 10px 0;">

### 💼 Manager’s Briefing: Denoising Steps = Cost

Every denoising step requires a **full forward pass** through the model. At 50 steps, that’s 50x the compute of a single classification.

**Key numbers:**
- Standard: 50 steps (our model)
- Production: 20–50 steps (Stable Diffusion)
- Fast: 1–4 steps (distilled models like SDXL Turbo)

**Questions to ask your vendor:**
1. *"How many denoising steps does your model use?"*
2. *"Do you offer a distilled (faster) version?"*
3. *"What’s the quality trade-off at fewer steps?"*

</div>

---

---

### 📋 Co-Pilot Reference Card (rows 1–3)

| Notebook | Component | What It Does | Co-Pilot Metaphor |
|----------|-----------|-------------|-------------------|
| NB1 | Raw Ingredients | Images = tensors of numbers; text = embedding vectors; cosine sim measures closeness | "Here are the parts of the car and the map" |
| NB2 | CLIP (Translator) | Learns shared embedding space; contrastive loss pulls matching pairs together | "The co-pilot learns to read the map" |
| **NB3** | **Diffusion (Engine)** | **Forward: add noise. Reverse: predict & remove noise. UNet preserves spatial structure.** | **"The driver learns to navigate by being dropped at random locations"** |

---